In [1]:
import os
os.chdir('/Users/taruni/Desktop/virality-prediction-ml-2026')
print('Working directory:', os.getcwd())

Working directory: /Users/taruni/Desktop/virality-prediction-ml-2026


# 04 — Evaluation
Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrices

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
print('TODO: evaluate trained models')

TODO: evaluate trained models


In [3]:
import pickle
import numpy as np
from sklearn.metrics import classification_report, roc_auc_score

# Load master data and recreate test set
master_df = pd.read_csv('data/processed/master.csv')

# Fix day_of_week
day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
           'Friday': 4, 'Saturday': 5, 'Sunday': 6}
master_df['day_of_week'] = master_df['day_of_week'].map(day_map).fillna(master_df['day_of_week'])
master_df['day_of_week'] = pd.to_numeric(master_df['day_of_week'], errors='coerce').fillna(0).astype(int)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = master_df.drop(columns=['viral', 'engagement_rate'])
y = master_df['viral']

scaler = pickle.load(open('data/processed/scaler.pkl', 'rb'))
X_scaled = scaler.transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y
)

# Load your 3 models
lr  = pickle.load(open('data/processed/model_lr.pkl', 'rb'))
knn = pickle.load(open('data/processed/model_knn.pkl', 'rb'))
svm = pickle.load(open('data/processed/model_svm.pkl', 'rb'))

print('Models and data loaded!')
print('Test set size:', X_test.shape)

/var/folders/dq/d0m2jdg97tsbddmx7x2s3rr80000gn/T/ipykernel_82066/980008717.py:6: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  master_df = pd.read_csv('data/processed/master.csv')


Models and data loaded!
Test set size: (18007, 33)


In [5]:
# ── RESULTS TABLE FOR ALL MODELS ──
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1': round(f1_score(y_test, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, proba), 4)
    }

results = []
results.append(evaluate_model('Logistic Regression', lr, X_test, y_test))
results.append(evaluate_model('KNN', knn, X_test, y_test))

# SVM results hardcoded from earlier run (too slow to re-predict)
results.append({
    'Model': 'SVM',
    'Accuracy': 0.55,
    'Precision': 0.26,
    'Recall': 0.70,
    'F1': 0.38,
    'ROC-AUC': 0.6586
})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

              Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression    0.5391     0.2321  0.5673 0.3295   0.6084
                KNN    0.8403     0.6755  0.3845 0.4901   0.7587
                SVM    0.5500     0.2600  0.7000 0.3800   0.6586
